# 🏭 Vishwakarma-Env — LLM Training on Factory Operations

This notebook trains a small LLM to manage Indian manufacturing plants using GRPO reinforcement learning.

**Runtime:** T4 GPU (free tier), ~30 minutes for 50 training steps

**What you'll see:** Reward curve going up as the LLM learns to respond to crises

In [ ]:
# Install dependencies
!pip install -q trl transformers torch accelerate datasets
!pip install -q git+https://github.com/YOUR-USERNAME/vishwakarma-env.git
# OR install locally:
# !pip install -q -e /content/vishwakarma_env

In [ ]:
# If installing from zip instead of GitHub:
import os
if not os.path.exists('/content/vishwakarma_env'):
    # Upload your vishwakarma_env_final.zip to Colab, then:
    !unzip -q vishwakarma_env_final.zip -d /content/
    !pip install -q -e /content/vishwakarma_env
    import sys
    sys.path.insert(0, '/content')
print('✓ Environment ready')

In [ ]:
# Verify environment works
import sys
sys.path.insert(0, '/content')

from vishwakarma_env.server.vishwakarma_environment import VishwakarmaEnvironment
from vishwakarma_env.models import VishwakarmaAction

env = VishwakarmaEnvironment(seed=0)
obs = env.reset()
obs2 = env.step(VishwakarmaAction(directive='run_normal', reasoning='test'))
print(f'✓ Environment works: reward={obs2.reward:.3f}')
print(f'  Target: {obs2.units_target_today} units/day')
print(f'  Budget: Rs{obs2.budget_today_INR:,}')

In [ ]:
import json
import torch
from vishwakarma_env.server.vishwakarma_environment import VishwakarmaEnvironment
from vishwakarma_env.models import VishwakarmaAction

def build_prompt(obs) -> str:
    machines_online = sum(1 for m in obs.machines if m.online)
    crisis_section = ''
    if obs.active_alerts:
        alert = obs.active_alerts[0]
        crisis_section = f"""
⚠ ACTIVE CRISIS [{alert.severity.upper()}]: {alert.message}
Options: {' | '.join(alert.resolution_options)}"""

    return f"""You manage a factory. Respond with JSON action.

STATUS (Step {obs.step}/16):
Machines: {machines_online}/{len(obs.machines)} | Workers: {obs.workers_present}/{obs.workers_total}
Stock: {obs.stock_tons:.1f}t ({obs.stock_days_remaining:.1f}d) | Units: {obs.units_produced_today}/{obs.units_target_today}
Cost: Rs{obs.cost_today_INR:,}/{obs.budget_today_INR:,}{crisis_section}

Respond ONLY with JSON:
{{"directive": "run_normal|call_maintenance|order_stock|authorize_overtime|call_contractor|accept_order|decline_order",
  "call_maintenance": false, "order_stock_tons": 0, "order_stock_supplier": "primary",
  "authorize_overtime_workers": 0, "call_contractors": 0, "adjust_production_rate": 1.0,
  "accept_emergency_order": false, "reroute_from": null, "reroute_to": null,
  "reasoning": "explain your decision"}}"""

def parse_action(text):
    text = text.replace('```json','').replace('```','').strip()
    try:
        s = text.find('{'); e = text.rfind('}')+1
        if s >= 0 and e > s:
            d = json.loads(text[s:e])
            return VishwakarmaAction(
                directive=d.get('directive','run_normal'),
                call_maintenance=bool(d.get('call_maintenance',False)),
                order_stock_tons=float(d.get('order_stock_tons',0)),
                order_stock_supplier=d.get('order_stock_supplier','primary'),
                authorize_overtime_workers=int(d.get('authorize_overtime_workers',0)),
                call_contractors=int(d.get('call_contractors',0)),
                adjust_production_rate=float(d.get('adjust_production_rate',1.0)),
                accept_emergency_order=bool(d.get('accept_emergency_order',False)),
                reroute_from=d.get('reroute_from'),
                reroute_to=d.get('reroute_to'),
                reasoning=d.get('reasoning',''),
            )
    except: pass
    return VishwakarmaAction(directive='run_normal', reasoning='parse error')

print('✓ Prompt builder and parser ready')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'  # 500M params, fits in T4 free tier
print(f'Loading {MODEL}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True
)
print(f'✓ Model loaded on {next(model.parameters()).device}')

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

N_TRAIN = 80   # number of training episodes
FACTORY = 'auto_components_pune'

# Build dataset
rows = []
for seed in range(N_TRAIN):
    env = VishwakarmaEnvironment(factory_id=FACTORY, seed=seed)
    obs = env.reset()
    rows.append({'prompt': build_prompt(obs), 'seed': str(seed)})
dataset = Dataset.from_list(rows)
print(f'✓ Dataset: {len(dataset)} training episodes')

# Reward function — runs full episode, returns total reward
def vishwakarma_reward(completions, prompts, seed, **kwargs):
    rewards = []
    for completion, s in zip(completions, seed):
        env = VishwakarmaEnvironment(factory_id=FACTORY, seed=int(s))
        obs = env.reset()
        total = 0.0
        action = parse_action(completion)
        for _ in range(16):
            obs = env.step(action)
            total += obs.reward
            if obs.done: break
            # Use same action for rest of episode (simplification for training)
            action = VishwakarmaAction(directive='run_normal', reasoning='')
        rewards.append(total)
    return rewards

config = GRPOConfig(
    output_dir='/content/vishwakarma-grpo',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=5,
    max_completion_length=200,
    num_generations=2,
    report_to='none',
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=vishwakarma_reward,
    args=config,
    train_dataset=dataset,
)

print('✓ Trainer ready. Starting training...')

In [ ]:
# TRAIN — watch the reward go up
# This is the screenshot you need for your README
trainer.train()

In [ ]:
# Plot the reward curve
import matplotlib.pyplot as plt

log = trainer.state.log_history
steps   = [x['step'] for x in log if 'loss' in x]
rewards = [x.get('reward', x.get('loss', 0)) for x in log if 'loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, rewards, 'b-o', linewidth=2, markersize=4)
plt.xlabel('Training Step')
plt.ylabel('Episode Reward')
plt.title('Vishwakarma-Env: LLM Learning to Manage a Factory')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/vishwakarma_training_curve.png', dpi=150)
plt.show()
print('✓ Saved to /content/vishwakarma_training_curve.png')
print('  Download this and add to your README!')

In [ ]:
# Evaluate trained model vs untrained
def evaluate(model, n_episodes=10):
    model.eval()
    rewards = []
    for seed in range(n_episodes):
        env = VishwakarmaEnvironment(factory_id=FACTORY, seed=seed+200)
        obs = env.reset()
        total = 0.0
        for _ in range(16):
            prompt = build_prompt(obs)
            inputs = tokenizer(prompt, return_tensors='pt',
                               truncation=True, max_length=400).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=150,
                                      do_sample=False,
                                      pad_token_id=tokenizer.eos_token_id)
            resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                     skip_special_tokens=True)
            action = parse_action(resp)
            obs    = env.step(action)
            total += obs.reward
            if obs.done: break
        rewards.append(total)
    return sum(rewards)/len(rewards)

trained_score = evaluate(trainer.model)
print(f'Trained model avg reward:   {trained_score:.1f}')
print(f'(Random baseline is ~15.4, smart agent is ~49.3)')
print(f'If trained > 20, your environment is working as a training signal!')